# Trading Recommender — Live Kalshi Earnings Mention Markets

## Strategy: prioritize trading freshly-opened markets
Fresh markets (has_prior_session=0) have the most uninformed crowd pricing
and the biggest edge for our model. We show all signals but flag FRESH markets.

## Bug fixes applied:
1. has_candle=0 for live markets (no real candle data) — matches training distribution
2. sector_peer_base_rate uses actual sector lookup from historical data
3. Liquidity filter: skip markets with volume < 10 AND no prior session
4. Momentum flags only fire with prior session data (no false SPIKEs)

## Filters applied in order:
- transcript_prob < 0.05: no historical signal -> NO_HISTORY
- volume < 10 AND fresh market: -> ILLIQUID
- momentum conflict with bet direction: -> MOMENTUM_CONFLICT
- edge < threshold: -> LOW_EDGE

In [1]:
import os,re,time,base64,datetime,pickle,requests,json
import numpy as np,pandas as pd,warnings
from pathlib import Path
from dotenv import load_dotenv
from cryptography.hazmat.primitives import serialization,hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.backends import default_backend
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
load_dotenv("k.env")

KALSHI_KEY_ID   = os.getenv('KALSHI_KEY_ID')
KALSHI_KEY_FILE = os.getenv('KALSHI_KEY_FILE','kalshi_key.key')
BASE_URL        = 'https://api.elections.kalshi.com/trade-api/v2'
READ_DELAY      = 0.06
TRANSCRIPT_DIR  = Path('transcripts')

try:
    with open('model_output/trading_config.json') as f:
        cfg = json.load(f)
    EDGE_THRESHOLD      = cfg['edge_threshold']
    KELLY_FRACTION      = cfg['kelly_fraction']
    MAX_KELLY           = cfg['max_kelly']
    MIN_TRANSCRIPT_PROB = cfg.get('min_transcript_prob', 0.05)
    MIN_VOLUME          = cfg.get('min_volume', 10)
    STAGE2_FEATS        = cfg['stage2_feats']
    print('Config loaded from trading_config.json')
except Exception as e:
    print(f'Config load failed: {e}. Using defaults.')
    EDGE_THRESHOLD      = 0.10
    KELLY_FRACTION      = 0.25
    MAX_KELLY           = 0.10
    MIN_TRANSCRIPT_PROB = 0.05
    MIN_VOLUME          = 10
    STAGE2_FEATS        = ['tx_prob','open_mid_filled','has_candle']

print(f'Edge threshold:  {EDGE_THRESHOLD}')
print(f'Kelly: {KELLY_FRACTION} fractional, {MAX_KELLY} max')
print(f'Min tx_prob: {MIN_TRANSCRIPT_PROB} | Min volume: {MIN_VOLUME}')

Config loaded from trading_config.json
Edge threshold:  0.05
Kelly: 0.25 fractional, 0.1 max
Min tx_prob: 0.05 | Min volume: 10


In [2]:
def load_private_key(fp):
    with open(fp,'rb') as f:
        return serialization.load_pem_private_key(f.read(),password=None,
                                                   backend=default_backend())
def sign_pss(pk,text):
    return base64.b64encode(pk.sign(
        text.encode('utf-8'),
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()),
                    salt_length=padding.PSS.DIGEST_LENGTH),
        hashes.SHA256())).decode('utf-8')
def make_headers(method,path):
    ts=str(int(datetime.datetime.now().timestamp()*1000))
    return {'KALSHI-ACCESS-KEY':KALSHI_KEY_ID,
            'KALSHI-ACCESS-SIGNATURE':sign_pss(private_key,ts+method.upper()+path.split('?')[0]),
            'KALSHI-ACCESS-TIMESTAMP':ts,'Content-Type':'application/json'}
def kalshi_get(path,params=None,retries=3):
    url=BASE_URL+path
    for attempt in range(retries):
        resp=requests.get(url,headers=make_headers('GET',path),params=params)
        time.sleep(READ_DELAY)
        if resp.status_code==200: return resp.json()
        elif resp.status_code==429: time.sleep(2**attempt)
        else: return None
    return None
def paginate(path,key,params=None):
    params=dict(params or {}); params['limit']=200
    results=[]; cursor=None
    for _ in range(100):
        if cursor: params['cursor']=cursor
        data=kalshi_get(path,params)
        if not data: break
        batch=data.get(key,[])
        results.extend(batch)
        cursor=data.get('cursor')
        if not cursor or not batch: break
    return results
private_key=load_private_key(KALSHI_KEY_FILE)
print('Auth ready.')

Auth ready.


In [3]:
# ── Load Stage 1 and Stage 2 models ───────────────────────────────────────────
with open('model_output/stage1_lgb.pkl','rb') as f:
    s1=pickle.load(f)
stage1=s1['model']; imp_stage1=s1['imputer']; TRANSCRIPT_FEATS=s1['features']

with open('model_output/stage2_lr.pkl','rb') as f:
    s2=pickle.load(f)
stage2=s2['model']

# Load features_expanded for sector_peer_base_rate lookup
df_hist = pd.read_csv('model_data/features_expanded.csv')

# Build sector peer base rate lookup: (sector, word, period) -> mean base_rate
sector_pbr = df_hist.groupby(['sector','word','period_num'])['base_rate'].mean().reset_index()

def get_sector_peer_bra(sector, word, target_period):
    """Get mean base_rate of same-sector companies for this word in PRIOR period."""
    prior = sector_pbr[
        (sector_pbr['sector']==sector) &
        (sector_pbr['word']==word) &
        (sector_pbr['period_num'] == target_period - 1)
    ]
    if len(prior) > 0:
        return float(prior['base_rate'].mean())
    return None

# Sector map — must match training
SECTOR_MAP = {
    'AAPL':'Tech','MSFT':'Tech','GOOGL':'Tech','META':'Tech','AMZN':'Tech',
    'NVDA':'Tech','AMD':'Tech','INTC':'Tech','AVGO':'Tech','QCOM':'Tech',
    'CRM':'Tech','ADBE':'Tech','SNOW':'Tech','CRWD':'Tech','PLTR':'Tech',
    'MRVL':'Tech','MU':'Tech','TSM':'Tech','MSTR':'Tech','NBIS':'Tech',
    'RDDT':'Tech','HOOD':'Tech','COIN':'Tech','RKLB':'Tech','ASTS':'Tech',
    'OKLO':'Tech','CRWV':'Tech','CRCL':'Tech',
    'JPM':'Finance','BAC':'Finance','GS':'Finance','MS':'Finance',
    'WFC':'Finance','C':'Finance','AXP':'Finance','MA':'Finance',
    'V':'Finance','BLK':'Finance','KKR':'Finance','MCO':'Finance',
    'TFC':'Finance','ALLY':'Finance','RY':'Finance','PYPL':'Finance','DKNG':'Finance',
    'MCD':'Consumer','SBUX':'Consumer','CMG':'Consumer','WMT':'Consumer',
    'TGT':'Consumer','COST':'Consumer','KR':'Consumer','PEP':'Consumer',
    'KO':'Consumer','NKE':'Consumer','LULU':'Consumer','ANF':'Consumer',
    'ULTA':'Consumer','DIS':'Consumer','NFLX':'Consumer','SPOT':'Consumer',
    'EA':'Consumer','NYT':'Consumer','SNAP':'Consumer',
    'CELH':'Consumer','WEN':'Consumer','WING':'Consumer','SFM':'Consumer',
    'CBRL':'Consumer','GME':'Consumer','AMC':'Consumer','CVNA':'Consumer',
    'VSCO':'Consumer','LEVI':'Consumer','M':'Consumer','PARA':'Consumer',
    'JNJ':'Healthcare','LLY':'Healthcare','ISRG':'Healthcare','HIMS':'Healthcare',
    'BA':'Industrials','GEV':'Industrials','AVAV':'Industrials','KTOS':'Industrials',
    'DE':'Industrials','GM':'Industrials','F':'Industrials','TSLA':'Industrials',
    'LCID':'Industrials','NIO':'Industrials','WGO':'Industrials',
    'AAL':'Travel','UAL':'Travel','ALK':'Travel','DAL':'Travel',
    'HLT':'Travel','MTN':'Travel','LYFT':'Travel','UBER':'Travel',
    'ABNB':'Travel','CAR':'Travel',
    'HD':'Retail','AZO':'Retail','DELL':'Retail','HPE':'Retail',
    'SHOP':'Retail','ROKU':'Retail','CHWY':'Retail','ACI':'Retail',
    'PG':'Retail','STZ':'Retail','FDX':'Retail',
}

print(f'Stage 1: {len(TRANSCRIPT_FEATS)} features')
print(f'Stage 2 feats: {STAGE2_FEATS}')
print(f'Stage 2 coefs: {dict(zip(STAGE2_FEATS, stage2.coef_[0].round(3)))}')
print(f'Intercept: {stage2.intercept_[0]:.3f}')
print(f'Sector peer lookup: {len(sector_pbr):,} entries')

Stage 1: 23 features
Stage 2 feats: ['tx_prob', 'open_mid_filled', 'has_candle']
Stage 2 coefs: {'tx_prob': 1.707, 'open_mid_filled': 2.388, 'has_candle': -0.28}
Intercept: -1.798
Sector peer lookup: 92,704 entries


In [4]:
# ── Word counting + transcript parsing ────────────────────────────────────────
def normalize_kalshi_word(word):
    if not isinstance(word,str): return []
    return [v.strip().lower() for v in re.split(r'\s*/\s*',word) if v.strip()]

def count_word_in_text(kalshi_word,text):
    if not isinstance(text,str) or not text: return 0
    tl=text.lower()
    return sum(len(re.findall(r'\b'+re.escape(v)+r'\b',tl))
               for v in normalize_kalshi_word(kalshi_word))

def parse_transcript_file(filepath):
    with open(filepath,'r',encoding='utf-8',errors='ignore') as f:
        content=f.read()
    lines=content.split('\n')
    ticker_m=re.search(r'Symbol:\s*(\S+)',content)
    ticker=ticker_m.group(1).strip() if ticker_m else filepath.stem.split('_')[0]
    period_m=re.search(r'Period:\s*(Q\d)\s*(\d{4})',content,re.IGNORECASE)
    quarter=int(period_m.group(1)[1]) if period_m else None
    year=int(period_m.group(2)) if period_m else None
    date_m=re.search(r'Date:\s*(\d{4}-\d{2}-\d{2})',content)
    earnings_date=pd.Timestamp(date_m.group(1)).date() if date_m else None
    body_start=0
    for i,line in enumerate(lines):
        s=line.strip()
        if s and all(c=='=' for c in s) and len(s)>=4:
            body_start=i+1; break
    parts=[]
    for line in lines[body_start:]:
        s=line.strip()
        if not s: continue
        m=re.match(r'^([A-Z][A-Za-z\s\.\-]+):\s*(.*)',s)
        if m:
            speech=m.group(2).strip()
            if speech: parts.append(speech)
        else:
            parts.append(s)
    mgmt=' '.join(parts)
    if len(mgmt)<500: mgmt=' '.join(l.strip() for l in lines[body_start:] if l.strip())
    return {'ticker':ticker,'year':year,'quarter':quarter,
            'earnings_date':earnings_date,'management_text':mgmt}

print('Loading transcripts...')
all_tx=[]
for d in sorted(TRANSCRIPT_DIR.iterdir()):
    if not d.is_dir(): continue
    for f in sorted(d.glob('*.txt')):
        try:
            p=parse_transcript_file(f)
            if p['year'] and p['quarter']: all_tx.append(p)
        except: pass

df_tx=pd.DataFrame(all_tx)
df_tx['earnings_date']=pd.to_datetime(df_tx['earnings_date'])
df_tx['period_num']=df_tx['year']*4+df_tx['quarter']
df_tx['sort_key']=df_tx['earnings_date'].fillna(
    pd.to_datetime(df_tx['period_num'].apply(
        lambda p:f'{p//4}-{(p%4)*3+1:02d}-01')))
tx_by_company={t:g.sort_values('sort_key').reset_index(drop=True)
               for t,g in df_tx.groupby('ticker')}

lly=tx_by_company['LLY'].iloc[-1]
print(f'Loaded {len(df_tx):,} | {df_tx["ticker"].nunique()} companies')
print(f'LLY Q{lly["quarter"]} {lly["year"]}: {len(lly["management_text"]):,} chars | '
      f'China={count_word_in_text("China",lly["management_text"])}')

Loading transcripts...
Loaded 7,224 | 123 companies
LLY Q4 2025: 59,551 chars | China=5


In [5]:
SERIES_TO_STOCK = {
    'KXEARNINGSMENTIONLLY':'LLY','KXEARNINGSMENTIONUAL':'UAL',
    'KXEARNINGSMENTIONAAPL':'AAPL','KXEARNINGSMENTIONINTC':'INTC',
    'KXEARNINGSMENTIONORCL':'ORCL','KXEARNINGSMENTIONMETA':'META',
    'KXEARNINGSMENTIONGOOGL':'GOOGL','KXEARNINGSMENTIONALIBABA':'BABA',
    'KXEARNINGSMENTIONCRCL':'CRCL','KXEARNINGSMENTIONCOST':'COST',
    'KXEARNINGSMENTIONAMZN':'AMZN','KXEARNINGSMENTIONSHOP':'SHOP',
    'KXEARNINGSMENTIONSNAP':'SNAP','KXEARNINGSMENTIONGME':'GME',
    'KXEARNINGSMENTIONROKU':'ROKU','KXEARNINGSMENTIONPEP':'PEP',
    'KXEARNINGSMENTIONHIMS':'HIMS','KXEARNINGSMENTIONJNJ':'JNJ',
    'KXEARNINGSMENTIONWING':'WING','KXEARNINGSMENTIONCRWV':'CRWV',
    'KXEARNINGSMENTIONDASH':'DASH','KXEARNINGSMENTIONPGR':'PGR',
    'KXEARNINGSMENTIONETOR':'ETOR','KXEARNINGSMENTIONDOCUSIGN':'DOCU',
    'KXEARNINGSMENTIONBA':'BA','KXEARNINGSMENTIONDIS':'DIS',
    'KXEARNINGSMENTIONVSCO':'VSCO','KXEARNINGSMENTIONTSLA':'TSLA',
    'KXEARNINGSMENTIONPSKY':'PARA','KXEARNINGSMENTIONC':'C',
    'KXEARNINGSMENTIONCMG':'CMG','KXEARNINGSMENTIONHLT':'HLT',
    'KXEARNINGSMENTIONKR':'KR','KXEARNINGSMENTIONALK':'ALK',
    'KXEARNINGSMENTIONPG':'PG','KXEARNINGSMENTIONBLK':'BLK',
    'KXEARNINGSMENTIONMSTR':'MSTR','KXEARNINGSMENTIONGS':'GS',
    'KXEARNINGSMENTIONRY':'RY','KXEARNINGSMENTIONMCD':'MCD',
    'KXEARNINGSMENTIONULTA':'ULTA','KXEARNINGSMENTIONRKLB':'RKLB',
    'KXEARNINGSMENTIONACI':'ACI','KXEARNINGSMENTIONAVGO':'AVGO',
    'KXEARNINGSMENTIONLEVI':'LEVI','KXEARNINGSMENTIONAAL':'AAL',
    'KXEARNINGSMENTIONSNOW':'SNOW','KXEARNINGSMENTIONMRVL':'MRVL',
    'KXEARNINGSMENTIONPYPL':'PYPL','KXEARNINGSMENTIONNVDA':'NVDA',
    'KXEARNINGSMENTIONCAVA':'CAVA','KXEARNINGSMENTIONNKE':'NKE',
    'KXEARNINGSMENTIONNFLX':'NFLX','KXBRKMENTION':'BRK-B',
    'KXEARNINGSMENTIONAMD':'AMD','KXEARNINGSMENTIONVZ':'VZ',
    'KXEARNINGSMENTIONDAL':'DAL','KXEARNINGSMENTIONHD':'HD',
    'KXEARNINGSMENTIONADBE':'ADBE','KXEARNINGSMENTIONF':'F',
    'KXEARNINGSMENTIONDELL':'DELL','KXEARNINGSMENTIONHOOD':'HOOD',
    'KXEARNINGSMENTIONMTN':'MTN','KXEARNINGSMENTIONAVAV':'AVAV',
    'KXEARNINGSMENTIONCAR':'CAR','KXEARNINGSMENTIONBAC':'BAC',
    'KXEARNINGSMENTIONISRG':'ISRG','KXEARNINGSMENTIONLUCID':'LCID',
    'KXEARNINGSMENTIONCBRL':'CBRL','KXEARNINGSMENTIONSTZ':'STZ',
    'KXEARNINGSMENTIONDKNG':'DKNG','KXEARNINGSMENTIONJPM':'JPM',
    'KXEARNINGSMENTIONNIO':'NIO','KXEARNINGSMENTIONWFC':'WFC',
    'KXEARNINGSMENTIONASTS':'ASTS','KXEARNINGSMENTIONCHWY':'CHWY',
    'KXEARNINGSMENTIONCRM':'CRM','KXEARNINGSMENTIONPLTR':'PLTR',
    'KXEARNINGSMENTIONMU':'MU','KXEARNINGSMENTIONTFC':'TFC',
    'KXEARNINGSMENTIONON':'ON','KXEARNINGSMENTIONWMT':'WMT',
    'KXEARNINGSMENTIONQCOM':'QCOM','KXEARNINGSMENTIONTSMC':'TSM',
    'KXEARNINGSMENTIONRDDT':'RDDT','KXEARNINGSMENTIONMA':'MA',
    'KXEARNINGSMENTIONNBIS':'NBIS','KXEARNINGSMENTIONV':'V',
    'KXEARNINGSMENTIONSPOT':'SPOT','KXEARNINGSMENTIONLYFT':'LYFT',
    'KXEARNINGSMENTIONKTOS':'KTOS','KXEARNINGSMENTIONAMC':'AMC',
    'KXEARNINGSMENTIONGEV':'GEV','KXEARNINGSMENTIONHPE':'HPE',
    'KXEARNINGSMENTIONKO':'KO','KXEARNINGSMENTIONMSFT':'MSFT',
    'KXEARNINGSMENTIONABNB':'ABNB','KXEARNINGSMENTIONFDX':'FDX',
    'KXEARNINGSMENTIONMCO':'MCO','KXEARNINGSMENTIONAXP':'AXP',
    'KXEARNINGSMENTIONMS':'MS','KXEARNINGSMENTIONCRWD':'CRWD',
    'KXEARNINGSMENTIONTGT':'TGT','KXEARNINGSMENTIONALLY':'ALLY',
    'KXEARNINGSMENTIONEA':'EA','KXEARNINGSMENTIONCOINBASE':'COIN',
    'KXEARNINGSMENTIONAZO':'AZO','KXEARNINGSMENTIONDE':'DE',
    'KXEARNINGSMENTIONKKR':'KKR','KXEARNINGSMENTIONGM':'GM',
    'KXMENTIONEARNAIR':'AIR','KXMENTIONEARNGIS':'GIS',
    'KXMENTIONEARNM':'M','KXMENTIONEARNOKLO':'OKLO',
    'KXMENTIONEARNWGO':'WGO','KXMENTIONEARNLULU':'LULU',
    'KXEARNINGSMENTIONCVNA':'CVNA','KXEARNINGSMENTIONSBUX':'SBUX',
    'KXEARNINGSMENTIONCELH':'CELH','KXEARNINGSMENTIONUBER':'UBER',
    'KXEARNINGSMENTIONWEN':'WEN','KXEARNINGSMENTIONSFM':'SFM',
    'KXEARNINGSMENTIONANF':'ANF','KXMENTIONEARNCCL':'CCL',
    'KXEARNINGSMENTIONNYT':'NYT',
}
MONTH_MAP={'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,
           'JUL':7,'AUG':8,'SEP':9,'OCT':10,'NOV':11,'DEC':12}
print(f'{len(SERIES_TO_STOCK)} companies')

125 companies


In [6]:
# ── Pull markets + compute features + recommend ───────────────────────────────
print('Pulling open markets...')
live_markets = []
seen = set()

for series, stock in SERIES_TO_STOCK.items():
    events = paginate('/events','events',
                      params={'series_ticker':series,'with_nested_markets':'true'})
    for event in events:
        et = event.get('event_ticker','')
        m = re.search(r'-(\d{2})([A-Z]{3})(\d{2})$', et)
        if m:
            cy=2000+int(m.group(1)); cm=MONTH_MAP.get(m.group(2),1); cd=int(m.group(3))
            try: call_date=pd.Timestamp(year=cy,month=cm,day=cd)
            except: call_date=None
            tq=(cm-1)//3+1; ty=cy
        else:
            call_date=None; tq=2; ty=2026

        for mkt in event.get('markets',[]):
            if mkt.get('status')!='active': continue
            ticker=mkt.get('ticker','')
            if not ticker or ticker in seen: continue
            seen.add(ticker)

            word = mkt.get('yes_sub_title') or mkt.get('custom_strike',{}).get('Word','')
            if not word: continue

            curr_bid=float(mkt.get('yes_bid_dollars',0) or 0)
            curr_ask=float(mkt.get('yes_ask_dollars',1) or 1)
            no_ask  =float(mkt.get('no_ask_dollars',1) or 1)
            no_bid  =float(mkt.get('no_bid_dollars',0) or 0)
            volume  =float(mkt.get('volume_fp',0) or 0)
            prev_bid=float(mkt.get('previous_yes_bid_dollars',0) or 0)
            prev_ask=float(mkt.get('previous_yes_ask_dollars',0) or 0)

            if prev_bid > 0 or prev_ask > 0:
                open_mid=(prev_bid+prev_ask)/2
                has_prior=True
            else:
                open_mid=(curr_bid+curr_ask)/2
                has_prior=False
            curr_mid=(curr_bid+curr_ask)/2
            momentum=curr_mid-open_mid if has_prior else 0.0

            live_markets.append({
                'market_ticker':ticker,'stock_ticker':stock,
                'event_ticker':et,'word':word,
                'yes_bid':curr_bid,'yes_ask':curr_ask,
                'no_bid':no_bid,'no_ask':no_ask,
                'volume':volume,'call_date':call_date,
                'target_year':ty,'target_quarter':tq,
                'live_open_mid':open_mid,'live_current_mid':curr_mid,
                'live_momentum':momentum,
                'has_prior_session':int(has_prior),
            })

df_live = pd.DataFrame(live_markets)
print(f'Open markets: {len(df_live)} | Companies: {df_live["stock_ticker"].nunique()}')
print(f'  Fresh markets (no prior session): {(df_live["has_prior_session"]==0).sum()}')
print(f'  Markets with prior session:       {(df_live["has_prior_session"]==1).sum()}')

# ── Build transcript features with real sector_peer_base_rate ─────────────────
COMMON={'revenue','growth','quarter','year','billion','million',
        'margin','profit','cost','market','customer','business'}

def build_features(stock_ticker,word,target_year,target_quarter):
    txs=tx_by_company.get(stock_ticker)
    if txs is None or len(txs)==0: return None
    tp=target_year*4+target_quarter
    past=txs[txs['period_num']<tp].tail(12)
    if len(past)==0: return None
    pm=past['management_text'].tolist()
    ps=[int(count_word_in_text(word,t)>0) for t in pm]
    pc=[count_word_in_text(word,t) for t in pm]
    n=len(past)
    def wr(w): s=ps[-w:]; return float(np.mean(s)) if s else 0.0
    def wc(w): return int(sum(pc[-w:]))
    r2=float(np.mean(pc[-2:])) if n>=2 else 0.0
    o2=float(np.mean(pc[-4:-2])) if n>=4 else 0.0
    vv=normalize_kalshi_word(word)

    # Real sector peer base rate lookup (prior period)
    sector = SECTOR_MAP.get(stock_ticker, 'Unknown')
    sector_bra = get_sector_peer_bra(sector, word, tp)
    if sector_bra is None:
        sector_bra = float(np.mean(ps))  # fallback to own base_rate

    return {
        'n_past':n,'base_rate':float(np.mean(ps)),'mean_count':float(np.mean(pc)),
        'max_count':int(max(pc)),'total_mentions':int(sum(pc)),
        'rate_last_1q':wr(1),'rate_last_2q':wr(2),'rate_last_4q':wr(4),
        'count_last_1q':wc(1),'count_last_2q':wc(2),'count_last_4q':wc(4),
        'said_last_1q':int(ps[-1]>0) if ps else 0,
        'said_last_2q':int(any(ps[-2:])) if len(ps)>=2 else 0,
        'said_last_4q':int(any(ps[-4:])) if len(ps)>=4 else 0,
        'trend_delta':r2-o2,'trend_ratio':r2/(o2+0.01),'is_trending_up':int(r2>o2),
        'mgmt_count_4q':wc(4),
        'word_n_variants':len(vv),'word_is_phrase':int(any(' ' in v for v in vv)),
        'word_char_len':len(word),'word_is_common_financial':int(any(v in COMMON for v in vv)),
        'sector_peer_base_rate':sector_bra,
    }

feat_rows=[]
for _,row in df_live.iterrows():
    f=build_features(row['stock_ticker'],row['word'],row['target_year'],row['target_quarter'])
    if f is not None:
        f['market_ticker']=row['market_ticker']
        feat_rows.append(f)

df_feats=pd.DataFrame(feat_rows)
df_rec=df_live.merge(df_feats,on='market_ticker',how='left')
df_rec=df_rec.dropna(subset=['base_rate']).copy()

# ── Stage 1: transcript probability ───────────────────────────────────────────
X1=imp_stage1.transform(df_rec[TRANSCRIPT_FEATS].fillna(0).values)
df_rec['tx_prob']=stage1.predict_proba(X1)[:,1]

# ── Stage 2: calibrated probability ───────────────────────────────────────────
# Use live_open_mid as the crowd opening signal (from previous session or current)
# This matches training: has_candle=1 rows used daily_open_mid
# We have an equivalent opening price from the API (previous_yes_bid/ask or current)
df_rec['open_mid_filled']=df_rec['live_open_mid']
df_rec['has_candle']=1

X2=df_rec[STAGE2_FEATS].values
df_rec['kalshi_prob']=stage2.predict_proba(X2)[:,1]

# ── Edges: trade against current live ask/bid ─────────────────────────────────
df_rec['edge_yes']=df_rec['kalshi_prob']-df_rec['yes_ask']
df_rec['edge_no']=(1-df_rec['kalshi_prob'])-df_rec['no_ask']

df_rec['kelly_yes']=(
    (df_rec['edge_yes']/(1-df_rec['yes_ask'].clip(0.01,0.99)))*KELLY_FRACTION
).clip(0,MAX_KELLY)
df_rec['kelly_no']=(
    (df_rec['edge_no']/df_rec['no_ask'].clip(0.01,0.99))*KELLY_FRACTION
).clip(0,MAX_KELLY)

# ── Momentum flags (only when we have prior session data) ─────────────────────
df_rec['momentum_flag']='NONE'
hp=df_rec['has_prior_session']==1
df_rec.loc[hp&(df_rec['live_momentum']>0.10),'momentum_flag']='TRENDING_YES'
df_rec.loc[hp&(df_rec['live_momentum']<-0.10),'momentum_flag']='TRENDING_NO'
df_rec.loc[hp&(df_rec['live_momentum'].abs()>0.20),'momentum_flag']='SPIKE'

# ── Company consensus score ───────────────────────────────────────────────────
def consensus(grp):
    n=len(grp)
    ny=(grp['edge_yes']>EDGE_THRESHOLD).sum()
    nn=(grp['edge_no']>EDGE_THRESHOLD).sum()
    return pd.Series({'consensus_yes':ny/n if n>0 else 0,
                      'consensus_no':nn/n if n>0 else 0,
                      'n_markets':n})
cons=df_rec.groupby('stock_ticker').apply(consensus).reset_index()
df_rec=df_rec.merge(cons,on='stock_ticker',how='left')

# ── Recommendation logic with all filters ─────────────────────────────────────
def recommend(row):
    # Filter 1: no transcript history
    if row['tx_prob']<MIN_TRANSCRIPT_PROB:
        return 'PASS','NO_HISTORY',0.0
    # Filter 2: illiquid markets (only matters for fresh markets)
    if row['volume']<MIN_VOLUME and row['has_prior_session']==0:
        return 'PASS','ILLIQUID',0.0

    ey=row['edge_yes']; en=row['edge_no']
    mf=row['momentum_flag']
    kp=row['kalshi_prob']
    om=row['live_open_mid']

    # REQUIRE_AGREE rule: model and crowd must agree on direction
    # This is the key insight: when model and crowd disagree, crowd wins 67% of the time
    # Only bet when BOTH the model AND the crowd's opening price point the same way
    if ey>=en and ey>EDGE_THRESHOLD:
        # BUY YES requires: kalshi_prob > 0.5 AND open price > 0.5
        if kp <= 0.5 or om <= 0.5:
            return 'PASS','DISAGREE_WITH_CROWD',0.0
        if mf=='TRENDING_NO': return 'PASS','MOMENTUM_CONFLICT',0.0
        conf='HIGH' if ey>0.15 else 'MEDIUM' if ey>0.10 else 'LOW'
        return 'BUY_YES',conf,row['kelly_yes']
    elif en>EDGE_THRESHOLD:
        # BUY NO requires: kalshi_prob < 0.5 AND open price < 0.5
        if kp >= 0.5 or om >= 0.5:
            return 'PASS','DISAGREE_WITH_CROWD',0.0
        if mf=='TRENDING_YES': return 'PASS','MOMENTUM_CONFLICT',0.0
        conf='HIGH' if en>0.15 else 'MEDIUM' if en>0.10 else 'LOW'
        return 'BUY_NO',conf,row['kelly_no']
    return 'PASS','LOW_EDGE',0.0

df_rec[['recommendation','confidence','kelly']]=df_rec.apply(
    lambda r:pd.Series(recommend(r)),axis=1)

# ── Display ───────────────────────────────────────────────────────────────────
COLS=['stock_ticker','word','recommendation','confidence',
      'has_prior_session','kalshi_prob','tx_prob','yes_ask','no_ask',
      'edge_yes','edge_no','kelly','base_rate','sector_peer_base_rate',
      'live_open_mid','live_current_mid','live_momentum',
      'momentum_flag','volume','consensus_yes','market_ticker']
COLS=[c for c in COLS if c in df_rec.columns]
df_out=df_rec[COLS].copy()
for c in ['kalshi_prob','tx_prob','yes_ask','no_ask','edge_yes','edge_no',
          'kelly','base_rate','sector_peer_base_rate',
          'live_open_mid','live_current_mid','live_momentum','consensus_yes']:
    if c in df_out: df_out[c]=pd.to_numeric(df_out[c],errors='coerce').round(3)

buy_yes=df_out[df_out['recommendation']=='BUY_YES'].sort_values('edge_yes',ascending=False)
buy_no =df_out[df_out['recommendation']=='BUY_NO'].sort_values('edge_no', ascending=False)

print('='*70)
print(f'KALSHI EARNINGS MENTION RECOMMENDATIONS')
print(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('='*70)
print(f'Analyzed: {len(df_rec)}')
print(f'  No history:          {(df_rec["confidence"]=="NO_HISTORY").sum()}')
print(f'  Illiquid:            {(df_rec["confidence"]=="ILLIQUID").sum()}')
print(f'  Disagree with crowd: {(df_rec["confidence"]=="DISAGREE_WITH_CROWD").sum()}')
print(f'  Momentum conflict:   {(df_rec["confidence"]=="MOMENTUM_CONFLICT").sum()}')
print(f'  Low edge:            {(df_rec["confidence"]=="LOW_EDGE").sum()}')
print(f'Actionable: {len(buy_yes)+len(buy_no)} (YES:{len(buy_yes)} NO:{len(buy_no)})')

if len(buy_yes)>0:
    print(f'\n--- BUY YES ({len(buy_yes)}) ---')
    print(buy_yes.to_string(index=False))
if len(buy_no)>0:
    print(f'\n--- BUY NO ({len(buy_no)}) ---')
    print(buy_no.to_string(index=False))

df_out.to_csv('model_output/live_recommendations.csv',index=False)
print('\nSaved -> model_output/live_recommendations.csv')

Pulling open markets...
Open markets: 281 | Companies: 20
  Fresh markets (no prior session): 0
  Markets with prior session:       281
KALSHI EARNINGS MENTION RECOMMENDATIONS
Generated: 2026-04-27 14:24
Analyzed: 281
  No history:          32
  Illiquid:            0
  Disagree with crowd: 88
  Momentum conflict:   4
  Low edge:            151
Actionable: 6 (YES:1 NO:5)

--- BUY YES (1) ---
stock_ticker  word recommendation confidence  has_prior_session  kalshi_prob  tx_prob  yes_ask  no_ask  edge_yes  edge_no  kelly  base_rate  sector_peer_base_rate  live_open_mid  live_current_mid  live_momentum momentum_flag  volume  consensus_yes                      market_ticker
        SPOT Apple        BUY_YES        LOW                  1        0.591    0.691      0.5    0.54     0.091   -0.131  0.045      0.583                  0.583           0.53              0.48          -0.05          NONE  342.71          0.133 KXEARNINGSMENTIONSPOT-26APR28-APPL

--- BUY NO (5) ---
stock_ticker       